# HTML to PNG converter
Run using virtual environment! Python 3.12 End 2 correct
This code works here because I did the following and there was already an existing version of chromium donwloaded on the server. So it was enough for me to download the web driver in the venv to get it to work. Since gpu server has no chromium, cannot download it, cant use web driver.

In [ ]:
import os
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
#from webdriver_manager.core.utils import ChromeType
#from webdriver_manager.chrome import ChromeDriverManager
#from webdriver_manager.core.os_manager import ChromeType
#from webdriver_manager.chrome import ChromeDriverManager


In [ ]:
import os

# Only move up if we're still in the utils/ directory
if os.getcwd().endswith('/utils'):
    os.chdir('..')

print(os.getcwd())  # should always show conformity-llms-facebook-posts

## Converting whole folders

In [ ]:
from pathlib import Path
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
import os
import time

def capture_screenshot(html_path, output_image="screenshot.png"):
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--window-size=800,1200")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.binary_location = "/usr/bin/chromium"

    driver = webdriver.Chrome(
        service=Service(os.path.expanduser("~/chromedriver")),
        options=chrome_options
    )

    try:
        file_url = "file://" + os.path.abspath(html_path)
        driver.get(file_url)
        time.sleep(2)
        driver.save_screenshot(output_image)
        print(f"Screenshot saved as {output_image}")
    finally:
        driver.quit()


def process_folder(label: str, html_dir: Path, png_dir: Path):
    """Process all HTML files in a folder, saving PNGs to the output dir."""
    if not html_dir.exists():
        print(f"[SKIP] Directory not found: {html_dir}")
        return

    html_files = sorted(html_dir.glob("*.html"))
    if not html_files:
        print(f"[SKIP] No HTML files found in: {html_dir}")
        return

    png_dir.mkdir(parents=True, exist_ok=True)
    print(f"\n[{label.upper()}] Processing {len(html_files)} file(s) from {html_dir}")
    '''
    for html_file in html_files:
        output_png = png_dir / (html_file.stem + ".png")

        if output_png.exists():
            print(f"  [SKIP] Already exists: {output_png.name}")
            continue

        print(f"  → {html_file.name}")
        capture_screenshot(str(html_file), str(output_png))
    '''
    for idx, html_file in enumerate(html_files):
        output_png = png_dir / (html_file.stem + ".png")
        
        if output_png.exists():
            print(f"  [SKIP] Already exists: {output_png.name}")
            continue
        
        print(f"  → {html_file.name}")
        capture_screenshot(str(html_file), str(output_png))
        
        # pause every 10 files
        if idx % 10 == 0:
            time.sleep(3)

BASE_DIR = Path().resolve()
METRICS_DIR = BASE_DIR / "spotify_pie_plot/pie_plot_posts/metrics_simple_plot/remy-ashford"
REACTION_VALUES = [10, 100, 1000, 10000, 100000, 1000000]

for label in ("correct", "incorrect"):
    for scale_value in REACTION_VALUES:
        process_folder(
            label=f"{label}/{scale_value}",
            #html_dir=METRICS_DIR / label / "html"/ "likes_only_noise"/ str(scale_value),
            #png_dir=METRICS_DIR / label / "PNGs" / "likes_only_noise" /str(scale_value),

            
            #html_dir=METRICS_DIR / label / "html"/ "realistic"/ str(scale_value),
            #png_dir=METRICS_DIR / label / "PNGs" / "realistic" /str(scale_value),

            html_dir=METRICS_DIR / label / "html"/ "likes_only"/ str(scale_value),
            png_dir=METRICS_DIR / label / "PNGs" / "likes_only" /str(scale_value),
    
        )

print("\nDone.")

In [ ]:
BASELINE_DIR = BASE_DIR / "spotify_pie_plot/pie_plot_posts/baselines_simple_plot"

for label in ("correct", "incorrect"):
    # remy-ashford baseline files sit directly in html/ (no subfolder)
    process_folder(
        label=f"{label}/remy-ashford",
        html_dir=BASELINE_DIR / label / "html",
        png_dir=BASELINE_DIR / label / "PNGs",
    )
    # news-outlet baseline files sit in a "news" subfolder
    process_folder(
        label=f"{label}/news",
        html_dir=BASELINE_DIR / label / "html" / "news",
        png_dir=BASELINE_DIR / label / "PNGs" / "news",
    )

print("\nDone.")

# Zipping folders to then download and upload onto GPU server

In [ ]:
import shutil
import os
from pathlib import Path

BASE_DIR = Path("/home/scc/miranda.barros-everett/conformity-llms-facebook-posts")

In [ ]:
folders = {
    #"correct_likes_only_noise": BASE_DIR / "spotify_pie_plot/pie_plot_posts/metrics/remy-ashford/correct/PNGs/likes_only_noise",
    "incorrect_likes_only_noise": BASE_DIR / "spotify_pie_plot/pie_plot_posts/metrics/remy-ashford/incorrect/PNGs/likes_only_noise",
    
    #"correct_uniform": BASE_DIR / "spotify_pie_plot/pie_plot_posts/metrics/remy-ashford/correct/PNGs/uniform",
    #"incorrect_uniform": BASE_DIR / "spotify_pie_plot/pie_plot_posts/metrics/remy-ashford/incorrect/PNGs/uniform",

    #"correct_realistic": BASE_DIR / "spotify_pie_plot/pie_plot_posts/metrics/remy-ashford/correct/PNGs/realistic",
    #"incorrect_realistic": BASE_DIR / "spotify_pie_plot/pie_plot_posts/metrics/remy-ashford/incorrect/PNGs/realistic",
}

for variant, folder_path in folders.items():
    zip_name = str(folder_path.parent / variant)  # saves as correct.zip / incorrect.zip inside their own folder
    shutil.make_archive(zip_name, 'zip', folder_path)
    print(f"✅ Zipped: {zip_name}.zip")

# Checking CPU usage and memory used by PNGs


In [ ]:

import psutil
import shutil

# RAM
ram = psutil.virtual_memory()
print(f"RAM total:     {ram.total / 1e9:.1f} GB")
print(f"RAM used:      {ram.used / 1e9:.1f} GB")
print(f"RAM available: {ram.available / 1e9:.1f} GB")
print(f"RAM usage:     {ram.percent}%")

# Disk
disk = shutil.disk_usage("/")
print(f"\nDisk total:    {disk.total / 1e9:.1f} GB")
print(f"Disk used:     {disk.used / 1e9:.1f} GB")
print(f"Disk free:     {disk.free / 1e9:.1f} GB")

# CPU
print(f"\nCPU usage:     {psutil.cpu_percent(interval=1)}%")

import os
from pathlib import Path

png_dir = Path("/home/scc/miranda.barros-everett/conformity-llms-facebook-posts/spotify_pie_plot/pie_plot_posts/metrics")

total_size = 0
count = 0
for f in png_dir.rglob("*.png"):
    total_size += f.stat().st_size
    count += 1

print(f"PNG files found: {count}")
print(f"Total size: {total_size / 1e6:.1f} MB")
print(f"Average size per file: {total_size / count / 1e6:.2f} MB" if count > 0 else "No files found")
print(f"Estimated total for 1200 files: {(total_size / count * 1200) / 1e6:.1f} MB" if count > 0 else "")

## Used for converting individual benchmarking html files 

In [ ]:
from pathlib import Path

def capture_screenshot(html_path, output_image="screenshot.png"):
    """
    Opens the given HTML file in a headless browser and captures a screenshot.

    Args:
        html_path (str): Path to the local HTML file.
        output_image (str): Path where the screenshot will be saved.
    """
    if Path(output_image).exists():
        print(f"[SKIP] Already exists: {Path(output_image).name}")
        return
        
    # Set Chrome options for headless mode
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--window-size=800,1200")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.binary_location = "/usr/bin/chromium"
    
    # Use WebDriver Manager to automatically install and manage ChromeDriver
    driver = webdriver.Chrome(
    service=Service(os.path.expanduser("~/chromedriver")),
    options=chrome_options
    )
    
    try:
        # Convert local HTML path to file URL
        file_url = "file://" + os.path.abspath(html_path)
        driver.get(file_url)
        
        # Wait for the page to load
        time.sleep(2)  # Adjust if necessary
        
        # Capture the full-page screenshot
        driver.save_screenshot(output_image)
        print(f"Screenshot saved as {output_image}")
    
    finally:
        driver.quit()

# Example usage
BASE_DIR = Path().resolve()

# Run

# CORRECT TEXTS
capture_screenshot(
    str(BASE_DIR / "spotify_pie_plot/pie_plot_posts/for_benchmarking/correct/html/gender_neutral_metrics_001_c.html"),
    str(BASE_DIR / "spotify_pie_plot/pie_plot_posts/for_benchmarking/correct/PNGs/gender_neutral_metrics_001_c.png")
)
capture_screenshot(
    str(BASE_DIR / "spotify_pie_plot/pie_plot_posts/for_benchmarking/correct/html/gender_neutral_baseline_001_c.html"),
    str(BASE_DIR / "spotify_pie_plot/pie_plot_posts/for_benchmarking/correct/PNGs/gender_neutral_baseline_001_c.png")
)
capture_screenshot(
    str(BASE_DIR / "spotify_pie_plot/pie_plot_posts/for_benchmarking/correct/html/ny_times_baseline_001_c.html"),
    str(BASE_DIR / "spotify_pie_plot/pie_plot_posts/for_benchmarking/correct/PNGs/ny_times_baseline_001_c.png")
)
capture_screenshot(
    str(BASE_DIR / "spotify_pie_plot/pie_plot_posts/for_benchmarking/correct/html/fox_news_baseline_001_c.html"),
    str(BASE_DIR / "spotify_pie_plot/pie_plot_posts/for_benchmarking/correct/PNGs/fox_news_baseline_001_c.png")
)
capture_screenshot(
    str(BASE_DIR / "spotify_pie_plot/pie_plot_posts/for_benchmarking/correct/html/reuters_baseline_001_c.html"),
    str(BASE_DIR / "spotify_pie_plot/pie_plot_posts/for_benchmarking/correct/PNGs/fox_news_baseline_001_c.png")
)

#################################################################################################
# INCORRECT TEXTS
###############################################################################################
capture_screenshot(
    str(BASE_DIR / "spotify_pie_plot/pie_plot_posts/for_benchmarking/incorrect/html/gender_neutral_metrics_001_i.html"),
    str(BASE_DIR / "spotify_pie_plot/pie_plot_posts/for_benchmarking/incorrect/PNGs/gender_neutral_metrics_001_i.png")
)
capture_screenshot(
    str(BASE_DIR / "spotify_pie_plot/pie_plot_posts/for_benchmarking/incorrect/html/gender_neutral_baseline_001_i.html"),
    str(BASE_DIR / "spotify_pie_plot/pie_plot_posts/for_benchmarking/incorrect/PNGs/gender_neutral_baseline_001_i.png")
)
capture_screenshot(
    str(BASE_DIR / "spotify_pie_plot/pie_plot_posts/for_benchmarking/incorrect/html/ny_times_baseline_001_i.html"),
    str(BASE_DIR / "spotify_pie_plot/pie_plot_posts/for_benchmarking/incorrect/PNGs/ny_times_baseline_001_i.png")
)
capture_screenshot(
    str(BASE_DIR / "spotify_pie_plot/pie_plot_posts/for_benchmarking/incorrect/html/fox_news_baseline_001_i.html"),
    str(BASE_DIR / "spotify_pie_plot/pie_plot_posts/for_benchmarking/incorrect/PNGs/fox_news_baseline_001_i.png")
)

capture_screenshot(
    str(BASE_DIR / "spotify_pie_plot/pie_plot_posts/for_benchmarking/incorrect/html/fox_news_baseline_001_i.html"),
    str(BASE_DIR / "spotify_pie_plot/pie_plot_posts/for_benchmarking/incorrect/PNGs/fox_news_baseline_001_i.png")
)

